# 15_analyze_docking — 도킹 점수 '크기 편향' 보정

**한 줄 요약:** 도킹 점수는 분자가 클수록 유리하므로, **원자 1개당 결합에너지(Ligand Efficiency)** 로 다시 매겨 '덩치빨'이 아닌 진짜 효율 좋은 후보를 가린다.
**용어:** Ligand Efficiency(LE) = −결합에너지 ÷ 무거운 원자 수 (클수록 효율적).
**큰 흐름:** ① 준비·읽기·함수 → ② 무거운 원자 수·LE 계산 → ③ 대조군 기준·유망군 → ④ 크기편향 표시·저장

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 + 결과 읽기 + 원자수 함수
도킹 점수표와 후보 정보를 읽고, 무거운 원자 수를 세는 함수들을 준비한다.

In [ ]:
import os
import json
import urllib.request
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

RESDIR = "data/docking/results"
SCORES = os.path.join(RESDIR, "docking_scores.csv")
MANIFEST = "data/docking/docking_manifest.csv"
REF_LIG = "data/docking/receptor/ref_ligand.pdb"
OUT = os.path.join(RESDIR, "docking_analysis.csv")

scores = pd.read_csv(SCORES)
man = pd.read_csv(MANIFEST).set_index("np_id")


def heavy_from_smiles(smi):
    m = Chem.MolFromSmiles(str(smi))
    return m.GetNumHeavyAtoms() if m else np.nan


def heavy_from_pdb(path):
    n = 0
    with open(path) as f:
        for ln in f:
            if ln[:6].strip() in ("ATOM", "HETATM"):
                elem = ln[76:78].strip() or ln[12:14].strip()
                if elem.upper() != "H":
                    n += 1
    return n


def pubchem_smiles(name):
    for prop in ("SMILES", "ConnectivitySMILES", "IsomericSMILES", "CanonicalSMILES"):
        try:
            u = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
                 f"{name}/property/{prop}/JSON")
            with urllib.request.urlopen(u, timeout=30) as r:
                d = json.load(r)["PropertyTable"]["Properties"][0]
            for k in (prop, "SMILES", "ConnectivitySMILES", "IsomericSMILES", "CanonicalSMILES"):
                if d.get(k):
                    return d[k]
        except Exception:
            continue
    return None

🔎 **코드 뜯어보기 (셀 1)**
- `def heavy_from_smiles(smi): m = Chem.MolFromSmiles(smi); return m.GetNumHeavyAtoms()` : **무거운 원자(수소 제외) 수** 세기.
- `def heavy_from_pdb(path):` : PDB 파일에서 원자 줄을 세어(수소 제외) 개수 계산.
- `scores.set_index("id")` / `pd.read_csv(...)` : 점수표·매니페스트 읽기(01에서 설명).

### 셀 2 — 무거운 원자 수 & Ligand Efficiency 계산
각 후보의 원자 수를 세고, 결합에너지를 원자 수로 나눠 효율(LE)을 계산한다.

In [ ]:
heavy = {}
for _, r in scores.iterrows():
    i = str(r["id"])
    if i in man.index:
        heavy[i] = heavy_from_smiles(man.loc[i, "canonical_smiles"])
    elif i == "REF_cocrystal":
        heavy[i] = heavy_from_pdb(REF_LIG)
    elif i == "BI-3231":
        smi = pubchem_smiles("BI-3231")
        heavy[i] = heavy_from_smiles(smi) if smi else np.nan

scores["n_heavy"] = scores["id"].map(lambda i: heavy.get(str(i), np.nan))
scores["LE"] = -scores["affinity"] / scores["n_heavy"]

🔎 **코드 뜯어보기 (셀 2)**
- `scores["id"].map(lambda i: heavy.get(i, np.nan))` : 각 후보 id에 대해 원자 수를 붙임(`.map`=값 변환, `lambda`=즉석 함수).
- `scores["LE"] = -scores["affinity"] / scores["n_heavy"]` : **Ligand Efficiency** = −결합에너지 ÷ 원자 수(음수 결합에너지라 −를 붙여 양수로).

### 셀 3 — 대조군 기준 + 유망 후보 가리기
알려진 저해제(대조군)의 LE를 기준선으로, 결합력·효율 둘 다 통과한 후보를 찾는다.

In [ ]:
ctrl = scores[scores.type == "control"]
le_thr = ctrl["LE"].min()
aff_thr = ctrl["affinity"].max()   # 더 약한 대조군(덜 음수)
print("=== 대조군(알려진 저해제) ===")
print(ctrl[["id", "affinity", "n_heavy", "LE"]].to_string(index=False))
print(f"\n문턱: affinity <= {aff_thr:.1f} 이고 LE >= {le_thr:.3f} 이면 '진짜 유망'")

cand = scores[scores.type == "candidate"].copy()
cand["beats_affinity"] = cand["affinity"] <= aff_thr
cand["beats_LE"] = cand["LE"] >= le_thr
cand["both"] = cand["beats_affinity"] & cand["beats_LE"]

out = scores.sort_values("LE", ascending=False)
out.to_csv(OUT, index=False)

print("\n=== Ligand Efficiency 재랭킹 (높을수록 원자당 효율적) ===")
show = out[["id", "type", "affinity", "n_heavy", "LE"]].head(15)
print(show.to_string(index=False))

promising = cand[cand["both"]].sort_values("LE", ascending=False)
print(f"\n>>> affinity·LE 둘 다 대조군 이상인 후보: {len(promising)}개")
if len(promising):
    print(promising[["id", "affinity", "n_heavy", "LE",
                     "active_prob", "max_sim_known"]].to_string(index=False))

🔎 **코드 뜯어보기 (셀 3)**
- `scores[scores.type == "control"]` : 대조군(알려진 저해제)만 고르기. `.min()`/`.max()`로 LE·affinity 기준선.
- `cand["both"] = cand["beats_affinity"] & cand["beats_LE"]` : 결합력·효율 **둘 다** 기준 통과했는지(`&`=그리고).
- `.sort_values("LE", ascending=False)` : 효율 높은 순 정렬.

### 셀 4 — 크기 편향 후보 표시 + 저장
결합력은 좋지만 효율은 낮은(덩치빨) 후보를 따로 표시하고 결과를 저장한다.

In [ ]:
size_inflated = cand[(cand["beats_affinity"]) & (~cand["beats_LE"])]
if len(size_inflated):
    print(f"\n[크기 편향 주의] affinity는 좋지만 LE 낮음(덩치빨 의심) {len(size_inflated)}개:")
    print(size_inflated[["id", "affinity", "n_heavy", "LE"]].to_string(index=False))
print(f"\n저장: {OUT}")

🔎 **코드 뜯어보기 (셀 4)**
- `cand[(cand["beats_affinity"]) & (~cand["beats_LE"])]` : 결합력은 좋지만(`beats_affinity`) 효율은 미달(`~beats_LE`) → **크기 편향(덩치빨)** 의심. `.to_csv(...)`=저장.